# Demo: Transforming Time-Series Data

Melbourne daily minimum temperatures (1981-1990). We'll apply rolling statistics, ADF test, and differencing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

UB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF", "Light Blue": "#D2DFFF", "Navy": "#031643"}
UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62", "Medium Gray": "#9C9FA8", "Gray": "#CECFD4", "Light Gray": "#F2F2F2", "White": "#FFFFFF"}
US = {"Orange": "#EE7622", "Yellow": "#F9DC5C", "Red": "#9C0D22", "Green": "#23CE6B"}

mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
df = pd.read_csv("../data/melbourne_temps.csv", parse_dates=["Date"])
df = df.set_index("Date").asfreq("D")
temps = df["Temp"]
print(f"Dates: {temps.index.min().date()} to {temps.index.max().date()}")
print(f"Observations: {len(temps)}")
temps.head()


## Rolling Statistics

In [ ]:
window = 30
rmean = temps.rolling(window).mean()
rstd = temps.rolling(window).std()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(temps.index, temps.values, color=UN["Light Gray"])
axes[0].plot(rmean.index, rmean.values, color=UN["Black"], label="30-day mean")
axes[0].set_ylabel("Temperature (C)", fontsize=16)
axes[0].legend()
axes[0].set_title("Rolling Mean", fontsize=18, fontweight="bold")
axes[0].tick_params(axis="both", labelsize=14)
axes[1].plot(rstd.index, rstd.values, color=UB["Medium Blue"])
axes[1].set_ylabel("Std dev (C)", fontsize=16)
axes[1].set_title("Rolling Std", fontsize=18, fontweight="bold")
axes[1].tick_params(axis="both", labelsize=14)
plt.tight_layout()
plt.show()


## The ADF Test

In [ ]:
result = adfuller(temps.dropna(), autolag="AIC")
print(f"ADF statistic: {result[0]:.4f}")
print(f"p-value:       {result[1]:.6f}")
print(f"Stationary:    {'yes' if result[1] < 0.05 else 'no'}")


## Differencing

In [ ]:
diffed = temps.diff().dropna()

result_diff = adfuller(diffed, autolag="AIC")
print(f"Differenced ADF p-value: {result_diff[1]:.6f}")
print(f"Stationary: {'yes' if result_diff[1] < 0.05 else 'no'}")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(temps.index, temps.values, color=UN["Black"])
axes[0].set_ylabel("Temperature (C)", fontsize=16)
axes[0].set_title("Original", fontsize=18, fontweight="bold")
axes[0].tick_params(axis="both", labelsize=14)
axes[1].plot(diffed.index, diffed.values, color=US["Green"])
axes[1].axhline(y=0, color=UN["Medium Gray"], linestyle="--", alpha=0.5)
axes[1].set_ylabel("Daily change (C)", fontsize=16)
axes[1].set_title("First-differenced", fontsize=18, fontweight="bold")
axes[1].tick_params(axis="both", labelsize=14)
plt.tight_layout()
plt.show()


## Train/Test Split

In [ ]:
test_days = 365
train, test = temps.iloc[:-test_days], temps.iloc[-test_days:]
print(f"Train: {len(train)} days ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test:  {len(test)} days ({test.index[0].date()} to {test.index[-1].date()})")
